In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.5200000000000001, 10: 0.6164999999999999, 20: 0.6439999999999999, 30: 0.6599999999999998, 40: 0.6809999999999999, 50: 0.7005, 60: 0.6925000000000001, 70: 0.692, 80: 0.7149999999999999, 90: 0.7295, 100: 0.7384999999999999, 110: 0.739, 120: 0.7405000000000002, 130: 0.7455, 140: 0.7445000000000002, 150: 0.7489999999999999, 160: 0.7499999999999998, 170: 0.7585000000000002, 180: 0.7705, 190: 0.7665, 200: 0.7614999999999998, 210: 0.767, 220: 0.764, 230: 0.7710000000000001, 240: 0.7765000000000001, 250: 0.7780000000000002, 260: 0.7775000000000003, 270: 0.7685, 280: 0.7815000000000001, 290: 0.7748717948717949, 300: 0.7748717948717948}
{0: 0.00796, 10: 0.00531775, 20: 0.005124, 30: 0.005759999999999998, 40: 0.004218999999999999, 50: 0.004589749999999999, 60: 0.003613749999999999, 70: 0.0019959999999999995, 80: 0.002895, 90: 0.00421975, 100: 0.0036677499999999996, 110: 0.003959, 120: 0.003949749999999999, 130: 0.0036197500000000006, 140: 0.0038297499999999985, 150: 0.003419, 160: 0.0027999